# White blood cell classification — final code walkthrough

This notebook is the 10-minute code walkthrough. Every code block below is printed **straight from `src/`**, the
code that produced the results, and every result is **loaded from the saved run outputs on Google Drive**. Nothing is
retrained except one short live run (section 5) that shows the training loop works.

| Section | What it shows | About |
|---|---|---|
| 0 | Setup | 1 min |
| 1 | Data | 0.5 min |
| 2 | Finding the leak: matching each image to its original photo | 2 min |
| 3 | The group split | 1 min |
| 4 | Models | 1 min |
| 5 | Training recipe + live run | 1.5 min |
| 6 | Validation comparison (3 seeds) | 1 min |
| 7 | Test results, full and clean subset | 1 min |
| 8 | Error analysis | 1 min |
| 9 | Grad-CAM | 0.5 min |
| 10 | Ablations: augmentation, aspect ratio | 0.5 min |

**Before the presentation:** Runtime → Change runtime type → T4 GPU, then Runtime → Run all once, so every output
is already on screen. During the walkthrough, scroll and explain; re-run single cells only if asked.

## 0. Setup

In [ ]:
import os, sys, json, inspect, importlib
from IPython.display import Image as IPImage, Markdown, display
import pandas as pd

REPO_URL = "https://github.com/Yasmin-maker1/Automated-Classification-of-White-Blood-Cell-Subtypes-Using-Convolutional-Neural-Networks.git"
if os.path.exists("/content/repo/.git"):
    %cd /content/repo
    !git pull -q
else:
    !git clone -q {REPO_URL} /content/repo
    %cd /content/repo
!git log -1 --format="code version: %h  %s  (%cd)"

from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/wbc_project"
SPLIT = f"{DRIVE}/group_split.json"                 # made by src/group_split.py
EXP   = f"{DRIVE}/experiments_groupsplit"           # made by src/run_experiments.py
GS    = f"{DRIVE}/group_split"                      # figures + tables from group_split.py
for p in (SPLIT, EXP, GS):
    print(("ok      " if os.path.exists(p) else "MISSING ") + p)

sys.path.insert(0, "/content/repo/src")
import data, model, train, evaluate, group_split, run_experiments, gradcam, error_analysis

def show(obj):
    """Print the real source code of a function or class from src/."""
    src = inspect.getsource(obj)
    file = inspect.getsourcefile(obj).split("/repo/")[-1]
    line = inspect.getsourcelines(obj)[1]
    display(Markdown(f"`{file}`, line {line}\n```python\n{src}\n```"))

%matplotlib inline

The images are needed for the live run (section 5) and the live Grad-CAM (section 9). If `data/` is missing,
this cell downloads it again from Kaggle (upload `kaggle.json` when asked).

In [ ]:
if not os.path.exists("data/dataset2-master"):
    from google.colab import files
    files.upload()   # kaggle.json
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !pip -q install kaggle
    !kaggle datasets download -d paultimothymooney/blood-cells -p data --unzip
train_dir, test_dir = data.find_split_dirs("data")
print("TRAIN:", train_dir, "\nTEST :", test_dir)

## 1. Data

Kaggle *Blood Cell Images* (Mooney, 2018): 9,957 TRAIN + 2,487 TEST images, 4 balanced classes, all 320×240.
Every image is a rotated/flipped/shifted copy of one of **366 original photos** (`dataset-master/JPEGImages`), which
is why the black corners appear, and why a random split leaks.

In [ ]:
from torchvision import datasets
for name, d in (("TRAIN", train_dir), ("TEST", test_dir)):
    ds = datasets.ImageFolder(d)
    print(name, {c: ds.targets.count(i) for i, c in enumerate(ds.classes)}, "total", len(ds))

import matplotlib.pyplot as plt
from PIL import Image
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, c in zip(axes, sorted(os.listdir(train_dir))):
    f = sorted(os.listdir(f"{train_dir}/{c}"))[0]
    ax.imshow(Image.open(f"{train_dir}/{c}/{f}")); ax.set_title(f"{c.lower()}  {f}", fontsize=9); ax.axis("off")
plt.show()

## 2. Finding the leak — `src/group_split.py`

**Midterm feedback:** the near-duplicate check was not rotation-aware, and validation was saturated.

**What changed:** instead of comparing augmented images with each other, every image is matched back to the
**original photo** it was made from. Each original is rendered in 144 views (2 flips × a rotation every 5°, with the
same black corners), and each image goes to the photo whose best view is most similar. The *margin* over the
second-best photo says how sure the match is.

In [ ]:
show(group_split.embed_original_views)
show(group_split.match_to_originals)

How good is the matching? The check is independent: does the matched photo's label in `labels.csv` agree with the image's class folder?

In [ ]:
s = json.load(open(SPLIT))
summ = s["summary"]
print(f"Label agreement overall: {summ['label_agreement']:.1%}   by class: {summ['label_agreement_by_class']}")
print(f"Unsure threshold (margin): {summ['min_margin']}    photos used: {summ['unsure']['photos_used']} of {summ['unsure']['photos_available']}")
bands = pd.DataFrame(summ["by_margin_band"]).T
bands["used"] = ["excluded" if float(b.split("-")[1]) <= summ["min_margin"] else "kept" for b in bands.index]
display(bands)
for f in ["random_matches.png", "test_overlap_examples.png", "weakest_kept_matches.png"]:
    print(f); display(IPImage(f"{GS}/{f}"))

## 3. The group split

Train and validation are split **by photo** with `StratifiedGroupKFold`, so no photo is on both sides. Unsure matches
are left out of train and validation, and a test image only counts as *clean* if its photo is not in train and its
match is sure. The official TEST folder is never used for any decision.

In [ ]:
show(group_split.group_train_val)
show(data.split_from_file)

In [ ]:
display(pd.read_csv(f"{GS}/per_class_sources.csv"))
v, t = summ["val_sharing_photo_with_train"], summ["test"]
print(f"Validation images whose photo is also in train: random split {v['random_split']:.1%}  ->  group split {v['group_split']:.1%}")
print(f"TEST: {t['n']} images from {t['n_photos']} photos; {t['n_photo_in_train']} ({t['n_photo_in_train']/t['n']:.1%}) "
      f"come from {t['photos_shared_with_train']} photos also in train; {t['n_unsure']} unsure -> clean subset {t['n_clean']} images")

## 4. Models — `src/model.py`

Same 4-way head replacement for every pretrained network; the head is always called `fc` or `classifier`, so one
function freezes any backbone.

In [ ]:
show(model.build_model)
show(model.set_backbone_trainable)

In [ ]:
rows = []
for name in model.MODEL_NAMES:
    m = model.build_model(name, 4, pretrained=False)
    model.set_backbone_trainable(m, False)
    p = model.count_parameters(m)
    rows.append({"model": name, "parameters (M)": round(p["total"] / 1e6, 2), "trainable when frozen": p["trainable"]})
display(pd.DataFrame(rows))

## 5. Training recipe — `src/train.py`, `src/run_experiments.py`

One recipe for every model: Adam, head lr 1e-3, backbone lr 1e-4, weight decay 1e-4, batch 64, 2 head-only epochs then
full fine-tuning, early stopping on validation loss (patience 3), best checkpoint = lowest validation loss. Only the
backbone changes between runs. The split is fixed (`--split-seed 42`); `--seed` (42, 43, 44) changes initialisation,
data order and augmentation.

In [ ]:
show(train.run_epoch)
show(run_experiments.build_grid)

In [ ]:
cfg = json.load(open(f"{EXP}/densenet169_ft_s42/config.json"))
keep = ["model", "epochs", "freeze_epochs", "batch_size", "lr", "backbone_lr_mult", "weight_decay", "patience",
        "seed", "split_seed", "split_file", "img_size", "resize", "no_augment", "amp"]
defaults = {"resize": "squash", "no_augment": False}   # runs made before these options existed used these
display(pd.Series({k: cfg["args"].get(k, defaults.get(k)) for k in keep}, name="densenet169_ft_s42 config"))
print("train / val / test images:", cfg["n_train"], cfg["n_val"], cfg["n_test"])

**Live proof that the pipeline runs end to end:** 1 epoch of the SmallCNN on the group split (about 1 minute on a T4). Not a reported result.

In [ ]:
!python src/train.py --data-root data --model smallcnn --epochs 1 --split-file {SPLIT} --amp --num-workers 2 \
    --out-dir /content/live_demo --run-name smallcnn_live_1epoch

## 6. Validation comparison (3 seeds each, mean ± sample std)

Selection rule, fixed before looking at the test set: **lowest mean validation loss**.

In [ ]:
summary = pd.read_csv(f"{EXP}/summary.csv")
display(summary[["configuration", "seeds", "params (M)", "val loss", "val acc", "best epoch", "min / run"]])
for run in ["smallcnn_scratch_s42", "resnet50_ft_s42", "densenet169_ft_s42", "resnet50_frozen_s42"]:
    print(run); display(IPImage(f"{EXP}/{run}/curves.png", width=800))

## 7. Test results — `src/evaluate.py`

Every saved model was evaluated **once** on the TEST folder (no decision changed afterwards). *Clean test* = the
2,205 test images whose source photo is not in train.

In [ ]:
show(evaluate.main)

In [ ]:
display(summary[["configuration", "val acc", "test acc", "test macro-F1", "clean test acc", "clean test macro-F1"]])
runs = pd.read_csv(f"{EXP}/summary_runs.csv")
display(runs[["run", "val_loss", "val_acc", "test_acc", "test_macro_f1", "clean_test_acc", "clean_test_macro_f1"]].round(4))
print("Literature: Praveen et al. (2021) report 90% subtype accuracy on this dataset (random split, so possibly leaky).")

## 8. Error analysis — `src/error_analysis.py`

In [ ]:
for run in ["smallcnn_scratch_s42", "densenet169_ft_s42"]:
    print("=====", run)
    print(open(f"{EXP}/{run}/eval/classification_report.txt").read())
    display(IPImage(f"{EXP}/{run}/eval/confusion_matrix.png", width=900))

The test set holds only ~113 independent photos. Counting per photo shows the errors come in whole photos:

In [ ]:
show(error_analysis.per_photo_table)

In [ ]:
spec = s["images"]
for run in ["smallcnn_scratch_s42", "densenet169_ft_s42"]:
    pred = pd.read_csv(f"{EXP}/{run}/eval/predictions.csv")
    print("=====", run); display(error_analysis.photo_summary(error_analysis.per_photo_table(pred, spec)))
display(IPImage(f"{EXP}/smallcnn_scratch_s42/eval/misclassified.png", width=900))

## 9. Grad-CAM — `src/gradcam.py`

Gradient-weighted activations of the last convolutional stage (Selvaraju et al., 2017). One live example, then the
saved grids: 4 correct (one per class) and 4 confident errors from 4 different photos.

In [ ]:
show(gradcam.GradCAM)

In [ ]:
from predictor import Predictor
pr = Predictor(f"{EXP}/smallcnn_scratch_s42/best.pt")
f = sorted(os.listdir(f"{test_dir}/MONOCYTE"))[0]
out = pr.predict(Image.open(f"{test_dir}/MONOCYTE/{f}"))
print("live:", f, "->", out["pred"], f"({out['confidence']:.2f})"); display(out["overlay"])
for run in ["smallcnn_scratch_s42", "densenet169_ft_s42"]:
    p = f"{EXP}/{run}/eval/gradcam_examples.png"
    if os.path.exists(p):
        print(run); display(IPImage(p))

## 10. Ablations (SmallCNN, same recipe, 3 seeds)

Promised at the midterm: augmentation on vs. off. Asked in the feedback: the 4:3 → 1:1 squash. `pad` adds black bars
instead of stretching, so nothing is distorted or cut off.

In [ ]:
show(data.resize_ops)

In [ ]:
abl = summary[summary.configuration.str.startswith("smallcnn")]
display(abl[["configuration", "seeds", "val loss", "val acc", "test macro-F1", "clean test macro-F1"]])
if len(abl) == 1:
    print("Ablation runs not found yet: run section 4c of colab_quickstart.ipynb first.")

## Limitations

* Validation has few photos per class (4 lymphocyte, 6 monocyte), so it is ~5 points more optimistic than test.
* ± is seed-to-seed spread on one split; the test set's ~113 photos make per-class numbers far less certain.
* Grad-CAM figures are a handful of examples: illustrations, not proof.
* 783 TRAIN-folder images whose source photo could not be identified with confidence were left out.
* Research prototype on one public dataset, not a clinical tool.